# Dropout Risk Exploratory Analysis

## Project Overview
Analyze student history and engagement data to identify patterns commonly associated with dropout risk, enabling early intervention strategies.

## Learning Objectives
- Identify early warning indicators of dropout risk
- Analyze how engagement, grades, and demographics relate to dropout
- Build risk profiles based on observable student characteristics
- Quantify the relative contribution of different risk factors

## Problem Statement
School administrators need to identify students at risk of dropping out early enough to intervene. This analysis explores historical data to find the most predictive patterns.

## Why This Project Matters
Student dropout has lasting social and economic consequences. Early identification allows schools to provide targeted support — mentoring, tutoring, or counseling — before a student disengages permanently.

## Dataset Overview
Synthetic dataset of ~2,000 students with enrollment history, grades, attendance, demographics, and dropout status.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 2000
age_at_enrollment = np.random.choice(range(14, 20), n, p=[0.10, 0.25, 0.28, 0.20, 0.12, 0.05])
gender = np.random.choice(['Male', 'Female'], n, p=[0.50, 0.50])
gpa_sem1 = np.clip(np.random.normal(2.8, 0.8, n), 0, 4.0).round(2)
gpa_sem2 = np.clip(gpa_sem1 + np.random.normal(-0.1, 0.3, n), 0, 4.0).round(2)
attendance_pct = np.clip(np.random.normal(80, 15, n), 20, 100).round(1)
disciplinary_actions = np.random.poisson(0.5, n)
disciplinary_actions = np.clip(disciplinary_actions, 0, 8)
free_lunch = np.random.choice(['Yes', 'No'], n, p=[0.42, 0.58])
parent_involvement = np.random.choice(['Low', 'Medium', 'High'], n, p=[0.30, 0.45, 0.25])
extracurricular = np.random.choice(['Yes', 'No'], n, p=[0.35, 0.65])
failed_courses = np.random.poisson(0.8, n)
failed_courses = np.clip(failed_courses, 0, 6)

# Dropout probability driven by factors
dropout_logit = (-3.0
                 - 0.8 * gpa_sem2
                 - 0.03 * attendance_pct
                 + 0.4 * disciplinary_actions
                 + 0.3 * failed_courses
                 + np.where(free_lunch == 'Yes', 0.3, 0)
                 + np.where(parent_involvement == 'Low', 0.5, np.where(parent_involvement == 'Medium', 0, -0.3))
                 + np.where(extracurricular == 'Yes', -0.4, 0)
                 + np.random.normal(0, 0.5, n))
dropout_prob = 1 / (1 + np.exp(-dropout_logit))
dropout = np.random.binomial(1, dropout_prob)

df = pd.DataFrame({
    'StudentID': [f'S{i:04d}' for i in range(n)],
    'AgeAtEnrollment': age_at_enrollment, 'Gender': gender,
    'GPA_Sem1': gpa_sem1, 'GPA_Sem2': gpa_sem2,
    'AttendancePct': attendance_pct, 'DisciplinaryActions': disciplinary_actions,
    'FreeLunch': free_lunch, 'ParentInvolvement': parent_involvement,
    'Extracurricular': extracurricular, 'FailedCourses': failed_courses,
    'Dropout': dropout
})
df['GPATrend'] = (df['GPA_Sem2'] - df['GPA_Sem1']).round(2)
print(f'Dataset: {df.shape}')
print(f'Dropout rate: {df["Dropout"].mean():.1%}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nDropout distribution:\n{df["Dropout"].value_counts()}')

## Dropout Rate Overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['Dropout'].value_counts().plot.pie(ax=axes[0], autopct='%1.1f%%', labels=['Retained', 'Dropped Out'],
                                       colors=['steelblue', 'coral'], startangle=90)
axes[0].set_title('Overall Dropout Rate')
axes[0].set_ylabel('')
dropout_by_gender = df.groupby('Gender')['Dropout'].mean()
dropout_by_gender.plot.bar(ax=axes[1], edgecolor='black', color=['steelblue', 'coral'])
axes[1].set_title('Dropout Rate by Gender')
axes[1].set_ylabel('Dropout Rate')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'dropout_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## GPA and Academic Indicators

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.boxplot(data=df, x='Dropout', y='GPA_Sem2', ax=axes[0])
axes[0].set_title('Sem 2 GPA by Dropout Status')
axes[0].set_xticklabels(['Retained', 'Dropped Out'])
sns.boxplot(data=df, x='Dropout', y='FailedCourses', ax=axes[1])
axes[1].set_title('Failed Courses by Dropout Status')
axes[1].set_xticklabels(['Retained', 'Dropped Out'])
sns.boxplot(data=df, x='Dropout', y='GPATrend', ax=axes[2])
axes[2].set_title('GPA Trend (Sem2 - Sem1) by Dropout')
axes[2].set_xticklabels(['Retained', 'Dropped Out'])
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'gpa_indicators.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'Mean GPA — Retained: {df[df["Dropout"]==0]["GPA_Sem2"].mean():.2f}, Dropped: {df[df["Dropout"]==1]["GPA_Sem2"].mean():.2f}')

## Attendance and Discipline

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='Dropout', y='AttendancePct', ax=axes[0])
axes[0].set_title('Attendance by Dropout Status')
axes[0].set_xticklabels(['Retained', 'Dropped Out'])
sns.boxplot(data=df, x='Dropout', y='DisciplinaryActions', ax=axes[1])
axes[1].set_title('Disciplinary Actions by Dropout Status')
axes[1].set_xticklabels(['Retained', 'Dropped Out'])
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'attendance_discipline.png'), dpi=100, bbox_inches='tight')
plt.show()

## Socioeconomic and Family Factors

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
df.groupby('FreeLunch')['Dropout'].mean().plot.bar(ax=axes[0], edgecolor='black', color='teal')
axes[0].set_title('Dropout Rate by Free Lunch')
axes[0].tick_params(axis='x', rotation=0)
pi_order = ['Low', 'Medium', 'High']
df.groupby('ParentInvolvement')['Dropout'].mean().reindex(pi_order).plot.bar(ax=axes[1], edgecolor='black', color='coral')
axes[1].set_title('Dropout Rate by Parent Involvement')
axes[1].tick_params(axis='x', rotation=0)
df.groupby('Extracurricular')['Dropout'].mean().plot.bar(ax=axes[2], edgecolor='black', color='steelblue')
axes[2].set_title('Dropout Rate by Extracurricular')
axes[2].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'socioeconomic_factors.png'), dpi=100, bbox_inches='tight')
plt.show()

## Risk Factor Summary

In [ ]:
factors = {
    'Low GPA (< 2.0)': df[df['GPA_Sem2'] < 2.0]['Dropout'].mean(),
    'Low Attendance (< 65%)': df[df['AttendancePct'] < 65]['Dropout'].mean(),
    'Disciplinary (>= 2)': df[df['DisciplinaryActions'] >= 2]['Dropout'].mean(),
    'Failed >= 2 courses': df[df['FailedCourses'] >= 2]['Dropout'].mean(),
    'Low Parent Involvement': df[df['ParentInvolvement'] == 'Low']['Dropout'].mean(),
    'No Extracurricular': df[df['Extracurricular'] == 'No']['Dropout'].mean(),
    'Overall baseline': df['Dropout'].mean()
}
risk_df = pd.DataFrame({'Factor': factors.keys(), 'DropoutRate': factors.values()})
risk_df = risk_df.sort_values('DropoutRate', ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(risk_df['Factor'], risk_df['DropoutRate'], edgecolor='black', color='coral')
ax.set_xlabel('Dropout Rate')
ax.set_title('Dropout Rate by Risk Factor')
ax.axvline(df['Dropout'].mean(), color='blue', ls='--', label='Baseline')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'risk_factors.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Low GPA** is the strongest single indicator — students with GPA < 2.0 have dramatically higher dropout rates
- **Attendance below 65%** is a strong early warning signal
- **Disciplinary actions** compound the risk — behavioral engagement matters
- **Parent involvement** has a clear protective effect — outreach programs to low-involvement families could help
- **Extracurricular participation** is associated with lower dropout — it signals engagement and belonging
- A **composite risk score** combining GPA, attendance, and discipline would catch most at-risk students

## Limitations
- Synthetic data with simplified causal structure
- No longitudinal tracking over multiple years
- No data on school environment, teacher quality, or peer effects
- No intervention outcome data
- Dropout is binary — doesn't capture temporary leaves or transfers

## How to Improve This Project
- Add longitudinal tracking for multi-year trend analysis
- Include peer network and school climate data
- Build predictive models with calibrated probabilities
- Add intervention tracking to measure program effectiveness
- Include mental health and motivation indicators

## Production Considerations
- Real-time early warning dashboards for counselors
- Automated risk scoring updated weekly
- Integration with student information systems
- Privacy-preserving reporting that protects student identity

## Common Mistakes
- Relying on a single indicator for risk assessment
- Not updating risk scores as new data arrives
- Labeling students as 'dropout risks' without providing support
- Ignoring the socioeconomic context behind academic indicators

## Mini Challenge / Exercises
1. What percentage of students with BOTH low GPA (< 2.0) AND low attendance (< 65%) dropped out?
2. Among students with high parent involvement, what is the dropout rate for those with 2+ failed courses?
3. Create a simple risk score (1 point per risk factor) and calculate dropout rate at each score level.
4. Which single factor, if improved, would reduce the dropout rate the most?

## Final Summary / Key Takeaways
- Dropout risk is predictable from early academic and engagement signals
- GPA, attendance, and discipline are the top three indicators
- Socioeconomic factors create baseline risk that needs systemic support
- Extracurricular involvement and parent engagement are protective factors
- Data-driven early warning systems can guide timely interventions